# Diagnóstico de calidad de datos: accidentalidad vial en Bogotá
**GA1-220501094-01-AA1-EV06** · Aprendiz: _tu nombre_ · Ficha: _tu ficha_

## Momento 1. Reconocimiento del proyecto y de los datasets

**Proyecto:** Accidentalidad vial en Bogotá.

**Pregunta analítica:** ¿A qué horas, en qué días y en qué localidades ocurren más accidentes de tránsito (y muertes por accidentes) en Bogotá, y qué medio de transporte está más involucrado?

| Dataset | Fuente | Información que contiene | Relación con el problema |
|---|---|---|---|
| `mortalidad` (`accidentes.csv`) | Secretaría Distrital de Salud, Datos Abiertos Bogotá | Víctimas fatales por accidentes de tránsito: localidad, año, casos, sexo, mes, día, rango de hora, clase de accidente, medio de transporte | Hora, día, zona y medio de transporte para accidentes **con víctimas fatales** |
| `siniestros` (hoja *Siniestros*) | Secretaría Distrital de Movilidad, Anuario de Siniestralidad (`BaseSiniestralidad.xlsx`) | Un registro por siniestro vial: fecha, hora, localidad, clase, gravedad | Hora, día y zona de **todos los siniestros** |
| `vehiculos` (hoja *Vehículos*) | Mismo archivo | Vehículos involucrados en cada siniestro | Tipo de vehículo más involucrado |
| `actor_vial` (hoja *Actor vial*) | Mismo archivo | Personas involucradas (peatón, conductor, ciclista, etc.) | Perfil de víctimas y actores |

Las tres hojas del Anuario se relacionan por el código del accidente. La hoja *Diccionario* describe cada variable.

**Aspectos de calidad a revisar:** valores faltantes, tipos de datos (números o fechas guardados como texto), categorías inconsistentes o con "sin dato", duplicados, y si las variables necesarias (hora, zona, vehículo) existen y son utilizables.

## Momento 2. Preparación del entorno y carga de datos

**Edita solo la celda siguiente:** escribe los nombres exactos de tus archivos CSV, que deben estar en `data/raw/`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

RAW = Path("../data/raw")          # el notebook vive en notebooks/

# Excel del Anuario de Siniestralidad (se convierte una sola vez a CSV, más rápido de leer)
EXCEL = RAW / "BaseSiniestralidad.xlsx"
HOJAS_EXCEL = {                    # nombre del dataset -> nombre de la hoja
    "siniestros": "Siniestros",
    "vehiculos":  "Vehículos",
    "actor_vial": "Actor vial",
}

# Archivos que se diagnostican (CSV)
ARCHIVOS = {
    "mortalidad": "accidentes.csv",
    "siniestros": "anuario_siniestros.csv",
    "vehiculos":  "anuario_vehiculos.csv",
    "actor_vial": "anuario_actor_vial.csv",
}
HOJAS = {}   # solo se usa si algún archivo de ARCHIVOS fuera Excel


In [2]:
def cargar_csv(ruta):
    """Prueba varias codificaciones y separadores hasta leer el CSV con más de una columna."""
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        for sep in (",", ";"):
            try:
                df = pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False)
                if df.shape[1] > 1:
                    return df, enc, sep
            except Exception:
                continue
    raise ValueError(f"No se pudo leer {ruta}")

def cargar_archivo(ruta, hoja=0):
    """Lee CSV (probando codificaciones/separadores) o Excel (.xlsx/.xls)."""
    if ruta.suffix.lower() in (".xlsx", ".xls"):
        print(f"  Hojas de {ruta.name}:", pd.ExcelFile(ruta).sheet_names)
        return pd.read_excel(ruta, sheet_name=hoja), "excel", f"hoja={hoja}"
    return cargar_csv(ruta)

# Conversión única del Excel a CSV (puede tardar varios minutos si el archivo es grande)
if not all((RAW / ARCHIVOS[n]).exists() for n in HOJAS_EXCEL):
    xl = pd.ExcelFile(EXCEL)
    print("Hojas del Excel:", xl.sheet_names)
    for nombre, hoja in HOJAS_EXCEL.items():
        destino = RAW / ARCHIVOS[nombre]
        if not destino.exists():
            print(f"Convirtiendo hoja '{hoja}' a {destino.name} ...")
            xl.parse(hoja).to_csv(destino, index=False, encoding="utf-8")

dfs = {}
for nombre, archivo in ARCHIVOS.items():
    df, enc, det = cargar_archivo(RAW / archivo, HOJAS.get(nombre, 0))
    dfs[nombre] = df
    print(f"{nombre:<15} {archivo}  |  {enc}, {det}  |  forma={df.shape}")

mortalidad      accidentes.csv  |  utf-8, ,  |  forma=(13138, 19)
siniestros      anuario_siniestros.csv  |  utf-8, ,  |  forma=(11573, 25)
vehiculos       anuario_vehiculos.csv  |  utf-8, ,  |  forma=(19974, 18)
actor_vial      anuario_actor_vial.csv  |  utf-8, ,  |  forma=(28029, 31)


In [3]:
# ¿Qué años y fechas cubre cada dataset?
for nombre, df in dfs.items():
    for c in df.columns:
        nombre_col = str(c).lower()
        if nombre_col in ("ano", "aa_acc", "aa_siniestro", "annoacc", "año"):
            a = pd.to_numeric(df[c], errors="coerce")
            print(f"{nombre:<12} {c}: {a.min():.0f} a {a.max():.0f}")
        if nombre_col.startswith("fecha"):
            f = pd.to_datetime(df[c], errors="coerce")
            print(f"{nombre:<12} {c}: {f.min()} a {f.max()} | fechas no convertibles: {int(f.isna().sum() - df[c].isna().sum())}")


mortalidad   ANO: 2015 a 2026
siniestros   Fecha_Siniestro: 2025-01-01 00:00:00 a 2025-12-31 00:00:00 | fechas no convertibles: 0
siniestros   AA_Siniestro: 2025 a 2025
vehiculos    Fecha_Siniestro: 2025-01-01 00:00:00 a 2025-12-31 00:00:00 | fechas no convertibles: 0
vehiculos    AA_Siniestro: 2025 a 2025
actor_vial   FechaSiniestro: 2025-01-01 00:00:00 a 2025-12-31 00:00:00 | fechas no convertibles: 0
actor_vial   Fecha_CambioGravedad: 2025-01-02 00:00:00 a 2026-03-17 00:00:00 | fechas no convertibles: 0


## Momento 3. Inspección de la estructura

In [4]:
for nombre, df in dfs.items():
    print("=" * 70)
    print(nombre.upper(), "| shape:", df.shape)
    print("Columnas:", list(df.columns))
    print("\nPrimeros registros:")
    display(df.head())
    print("Últimos registros:")
    display(df.tail())

MORTALIDAD | shape: (13138, 19)
Columnas: ['_id', 'CODIGO_LOCALIDAD', 'ANO', 'casos', 'Sexo', 'MES_DEL_HECHO', 'DIA_DEL_HECHO', 'RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_', 'PERTENENCIA_GRUPAL', 'PAIS_DE_NACIMIENTO', 'CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE', 'CIRCUNSTANCIA_DEL_HECHO_DETALLADA', 'CONDICION_DE_LA_VICTIMA_AT_', 'MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE', 'OBJETO_DE_COLISION', 'LOCALIDAD', 'ANCESTRO_RACIAL', 'PERTENENCIA_ETNICA', 'CICLO_VITAL']

Primeros registros:


,_id,CODIGO_LOCALIDAD,ANO,casos,Sexo,MES_DEL_HECHO,DIA_DEL_HECHO,RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_,PERTENENCIA_GRUPAL,PAIS_DE_NACIMIENTO,CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE,CIRCUNSTANCIA_DEL_HECHO_DETALLADA,CONDICION_DE_LA_VICTIMA_AT_,MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE,OBJETO_DE_COLISION,LOCALIDAD,ANCESTRO_RACIAL,PERTENENCIA_ETNICA,CICLO_VITAL
0,12387,8,2015,1,Hombre,abril,sábado,00:00 a 02:59,Otro,Colombia,Choque con objeto fijo o en movimiento,Embriaguez (Alcohólica y no alcohólica),Conductor,Automóvil,Objeto fijo,Kennedy,Mestizo,Sin información,(18 a 28) Juventud
1,12388,0,2015,1,Hombre,septiembre,sábado,15:00 a 17:59,Ninguno,Colombia,Choque con otro vehículo,Sin información,Conductor,Automóvil,Automóvil,Bogotá,Mestizo,Sin información,(18 a 28) Juventud
2,12389,0,2015,1,Hombre,octubre,viernes,00:00 a 02:59,Ninguno,Colombia,Choque con objeto fijo o en movimiento,Otras circunstancias no especificadas,Pasajero,Automóvil,Objeto fijo,Bogotá,Mestizo,Sin información,(29 a 59) Adultez
3,12390,12,2015,1,Hombre,noviembre,domingo,03:00 a 05:59,Ninguno,Colombia,Choque con objeto fijo o en movimiento,Sin información,Pasajero,Automóvil,Objeto fijo,Barrios Unidos,Mestizo,Sin información,(29 a 59) Adultez
4,12391,6,2015,1,Hombre,octubre,viernes,00:00 a 02:59,Ninguno,Colombia,Choque con objeto fijo o en movimiento,Otras circunstancias no especificadas,Pasajero,Automóvil,Objeto fijo,Tunjuelito,Mestizo,Sin información,(29 a 59) Adultez


Últimos registros:


,_id,CODIGO_LOCALIDAD,ANO,casos,Sexo,MES_DEL_HECHO,DIA_DEL_HECHO,RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_,PERTENENCIA_GRUPAL,PAIS_DE_NACIMIENTO,CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE,CIRCUNSTANCIA_DEL_HECHO_DETALLADA,CONDICION_DE_LA_VICTIMA_AT_,MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE,OBJETO_DE_COLISION,LOCALIDAD,ANCESTRO_RACIAL,PERTENENCIA_ETNICA,CICLO_VITAL
13133,25520,0,2026,1,Mujer,abril,viernes,Sin información,Sin información,Colombia,Sin información,Sin información,Sin información,Sin información,Sin información,Bogotá,Sin Información,Sin información,(Más de 60) Adulto Mayor
13134,25521,2,2026,1,Hombre,mayo,sábado,Sin información,Ninguno,Colombia,Sin información,Exceso de velocidad,Conductor,Sin información,Sin información,Chapinero,Sin Información,Sin Pertenencia Étnica,(29 a 59) Adultez
13135,25522,0,2026,1,Hombre,mayo,sábado,Sin información,Ninguno,Colombia,Sin información,Exceso de velocidad,Conductor,Sin información,Sin información,Bogotá,Sin Información,Sin Pertenencia Étnica,(29 a 59) Adultez
13136,25523,0,2026,1,Hombre,mayo,jueves,Sin información,Ninguno,Colombia,Sin información,Exceso de velocidad,Conductor,Sin información,Sin información,Bogotá,Sin Información,Sin Pertenencia Étnica,(18 a 28) Juventud
13137,25524,19,2026,1,Mujer,abril,viernes,Sin información,Sin información,Colombia,Sin información,Sin información,Sin información,Sin información,Sin información,Ciudad Bolívar,Sin Información,Sin información,(Más de 60) Adulto Mayor


SINIESTROS | shape: (11573, 25)
Columnas: ['Codigo_Siniestro', 'Formulario', 'Longitud', 'Latitud', 'Direccion', 'Fecha_Siniestro', 'AA_Siniestro', 'MM_Siniestro', 'DD_Mes_Siniestro', 'Dia_Semana_Siniestro', 'Hora_Siniestro', 'Min_Siniestro', 'Localidad', 'Clase_Siniestro', 'Elemento_Choque', 'Tipo_Objeto_Fijo', 'Gravedad_indicador_30d', 'Con_Bicicleta', 'Con_Carga', 'Con_Moto', 'Con_Peaton', 'Con_Troncal', 'Con_Alimentador', 'Con_Zonal', 'Con_Padron_Dual']

Primeros registros:


,Codigo_Siniestro,Formulario,Longitud,Latitud,Direccion,Fecha_Siniestro,AA_Siniestro,MM_Siniestro,DD_Mes_Siniestro,Dia_Semana_Siniestro,Hora_Siniestro,Min_Siniestro,Localidad,Clase_Siniestro,Elemento_Choque,Tipo_Objeto_Fijo,Gravedad_indicador_30d,Con_Bicicleta,Con_Carga,Con_Moto,Con_Peaton,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Padron_Dual
0,11335436,A001717267,"-74,13660797754","4,58396118246",KR 32 51 B SUR 02,2025-09-02,2025,Septiembre,2,martes,13,5,TUNJUELITO,Choque,Vehículo,NaN,Con Heridos,SI,NaN,SI,NaN,NaN,NaN,NaN,NaN
1,11335446,A001715889,"-74,10750215766","4,61761509057",TV 42 5 B 02,2025-09-03,2025,Septiembre,3,miércoles,10,30,PUENTE ARANDA,Choque,Vehículo,NaN,Con Heridos,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN
2,11335449,A001717256,"-74,11973444407","4,60537371657",KR 50 17 SUR 02,2025-09-02,2025,Septiembre,2,martes,15,3,PUENTE ARANDA,Choque,Vehículo,NaN,Con Heridos,NaN,NaN,SI,NaN,NaN,NaN,SI,NaN
3,11335455,A001716936,"-74,07096848165","4,76241455065",CL 170 80 02,2025-09-02,2025,Septiembre,2,martes,12,40,SUBA,Choque,Vehículo,NaN,Con Heridos,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN
4,11335456,A001717272,"-74,13899002037","4,56025493203",CL 61 SUR 18 C 02,2025-09-02,2025,Septiembre,2,martes,14,1,CIUDAD BOLÍVAR,Choque,Vehículo,NaN,Con Heridos,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN


... (6 outputs left out)

ACTOR_VIAL | shape: (28029, 31)
Columnas: ['Codigo_Accidentado', 'Codigo_Siniestro', 'Formulario', 'Codigo_Vehiculo', 'Ccodigo_Victima', 'FechaSiniestro', 'AnnoSiniestro', 'mesSiniestro', 'DD_Mes_Siniestro', 'Dia_Semana_Siniestro', 'hourSiniestro', 'minuSiniestro', 'Localidad', 'Edad', 'Sexo', 'Muerte_Posterior', 'Fecha_CambioGravedad', 'Gravedad_Indicador_30d', 'Condicion', 'Condicion_Especifica', 'Tipo_SITP', 'Con_Bicicleta', 'Con_Carga', 'Con_Moto', 'Con_Peaton', 'Con_Troncal', 'Con_Alimentador', 'Con_Zonal', 'Con_Padron_Dual', 'victimaC', 'asociadoC']

Primeros registros:


,Codigo_Accidentado,Codigo_Siniestro,Formulario,Codigo_Vehiculo,Ccodigo_Victima,FechaSiniestro,AnnoSiniestro,mesSiniestro,DD_Mes_Siniestro,Dia_Semana_Siniestro,hourSiniestro,minuSiniestro,Localidad,Edad,Sexo,Muerte_Posterior,Fecha_CambioGravedad,Gravedad_Indicador_30d,Condicion,Condicion_Especifica,Tipo_SITP,Con_Bicicleta,Con_Carga,Con_Moto,Con_Peaton,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Padron_Dual,victimaC,asociadoC
0,14458158,11334975,A001716592,1.0,2,2025-08-21,2025,Agosto,21,jueves,18,20,Chapinero,33.0,Femenino,N,NaN,HERIDO,MOTOCICLISTA,Pasajero Motocicleta,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,Pasajeros de motocicleta,Vehículos livianos
1,14458203,11334976,A001716243,1.0,0,2025-08-21,2025,Agosto,21,jueves,9,0,Los Mártires,53.0,Masculino,N,NaN,HERIDO,MOTOCICLISTA,Motociclista,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,Motociclistas,Vehículos livianos
2,14458204,11334976,A001716243,2.0,1,2025-08-21,2025,Agosto,21,jueves,9,0,Los Mártires,60.0,Masculino,N,NaN,ILESO,CONDUCTOR,Cond Servicio Especial,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,Ocupantes de vehículos livianos,Motocicletas
3,14458160,11334977,A001715710,1.0,0,2025-08-20,2025,Agosto,20,miércoles,12,30,Kennedy,37.0,Masculino,N,NaN,ILESO,CONDUCTOR,Cond Livianos,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,Ocupantes de vehículos livianos,Peatones
4,14458161,11334977,A001715710,NaN,1,2025-08-20,2025,Agosto,20,miércoles,12,30,Kennedy,68.0,Masculino,N,NaN,HERIDO,PEATÓN,Peatón,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,Peatones,Vehículos livianos


Últimos registros:


,Codigo_Accidentado,Codigo_Siniestro,Formulario,Codigo_Vehiculo,Ccodigo_Victima,FechaSiniestro,AnnoSiniestro,mesSiniestro,DD_Mes_Siniestro,Dia_Semana_Siniestro,hourSiniestro,minuSiniestro,Localidad,Edad,Sexo,Muerte_Posterior,Fecha_CambioGravedad,Gravedad_Indicador_30d,Condicion,Condicion_Especifica,Tipo_SITP,Con_Bicicleta,Con_Carga,Con_Moto,Con_Peaton,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Padron_Dual,victimaC,asociadoC
28024,14458923,11335308,A001716369,1.0,0,2025-08-23,2025,Agosto,23,sábado,5,20,Kennedy,37.0,Masculino,N,NaN,ILESO,CONDUCTOR,Cond Carga,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,Ocupantes de transporte de carga,Motocicletas
28025,14458924,11335308,A001716369,2.0,1,2025-08-23,2025,Agosto,23,sábado,5,20,Kennedy,30.0,Masculino,N,NaN,HERIDO,MOTOCICLISTA,Motociclista,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,Motociclistas,Transporte de carga
28026,14458925,11335308,A001716369,2.0,2,2025-08-23,2025,Agosto,23,sábado,5,20,Kennedy,32.0,Femenino,S,2025-08-26,MUERTO,MOTOCICLISTA,Pasajero Motocicleta,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,Pasajeros de motocicleta,Transporte de carga
28027,14458912,11335309,A001716360,1.0,0,2025-08-22,2025,Agosto,22,viernes,7,25,Kennedy,31.0,Masculino,N,NaN,ILESO,CONDUCTOR,Cond Carga,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,Ocupantes de transporte de carga,Motocicletas
28028,14458913,11335309,A001716360,2.0,1,2025-08-22,2025,Agosto,22,viernes,7,25,Kennedy,35.0,Femenino,N,NaN,HERIDO,MOTOCICLISTA,Motociclista,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,Motociclistas,Transporte de carga


In [5]:
for nombre, df in dfs.items():
    print("=" * 70)
    print(nombre.upper(), "- dtypes")
    print(df.dtypes)
    print()
    df.info()

MORTALIDAD - dtypes
_id                                        int64
CODIGO_LOCALIDAD                           int64
ANO                                        int64
casos                                      int64
Sexo                                         str
MES_DEL_HECHO                                str
DIA_DEL_HECHO                                str
RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_          str
PERTENENCIA_GRUPAL                           str
PAIS_DE_NACIMIENTO                           str
CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE      str
CIRCUNSTANCIA_DEL_HECHO_DETALLADA            str
CONDICION_DE_LA_VICTIMA_AT_                  str
MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE         str
OBJETO_DE_COLISION                           str
LOCALIDAD                                    str
ANCESTRO_RACIAL                              str
PERTENENCIA_ETNICA                           str
CICLO_VITAL                                  str
dtype: object

<class 'pandas.DataFrame'>
RangeIn

In [6]:
for nombre, df in dfs.items():
    print("=" * 70)
    print(nombre.upper(), "- describe() numéricas")
    if len(df.select_dtypes("number").columns):
        display(df.describe())
    else:
        print("No hay columnas numéricas (todas se leyeron como texto).")

    # Variables de texto (en pandas 3 el tipo es 'str'; en versiones anteriores, 'object')
    texto = df.select_dtypes(exclude="number")
    print(nombre.upper(), "- describe() de texto")
    if texto.shape[1]:
        display(texto.describe())

MORTALIDAD - describe() numéricas


,_id,CODIGO_LOCALIDAD,ANO,casos
count,13138.000000,13138.000000,13138.000000,13138.0
mean,18955.500000,5.086162,2020.488507,1.0
std,3792.758253,6.366831,3.430702,0.0
min,12387.000000,0.000000,2015.000000,1.0
25%,15671.250000,0.000000,2017.000000,1.0
50%,18955.500000,0.500000,2021.000000,1.0
75%,22239.750000,9.000000,2024.000000,1.0
max,25524.000000,21.000000,2026.000000,1.0


MORTALIDAD - describe() de texto


,Sexo,MES_DEL_HECHO,DIA_DEL_HECHO,RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_,PERTENENCIA_GRUPAL,PAIS_DE_NACIMIENTO,CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE,CIRCUNSTANCIA_DEL_HECHO_DETALLADA,CONDICION_DE_LA_VICTIMA_AT_,MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE,OBJETO_DE_COLISION,LOCALIDAD,ANCESTRO_RACIAL,PERTENENCIA_ETNICA,CICLO_VITAL
count,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138,13138
unique,2,25,15,18,38,12,13,43,4,27,28,22,9,6,7
top,Hombre,mayo,sábado,Sin información,Ninguno,Colombia,Atropello,Sin información,Conductor,No aplica,Motocicleta,Bogotá,Mestizo,Sin Pertenencia Étnica,(29 a 59) Adultez
freq,10310,1014,1802,4160,9918,12706,5572,7582,6270,5086,2718,6569,9612,7938,5400


SINIESTROS - describe() numéricas


,Codigo_Siniestro,AA_Siniestro,DD_Mes_Siniestro,Hora_Siniestro,Min_Siniestro
count,1.157300e+04,11573.0,11573.000000,11573.000000,11573.000000
mean,1.115171e+07,2025.0,15.755552,12.687462,26.095826
std,2.884698e+05,0.0,8.753595,5.940835,17.399667
min,1.061423e+07,2025.0,1.000000,0.000000,0.000000
25%,1.096754e+07,2025.0,8.000000,8.000000,10.000000
50%,1.133335e+07,2025.0,16.000000,13.000000,30.000000
75%,1.133646e+07,2025.0,23.000000,18.000000,40.000000
max,1.134250e+07,2025.0,31.000000,23.000000,59.000000


... (6 outputs left out)

ACTOR_VIAL - describe() numéricas


,Codigo_Accidentado,Codigo_Siniestro,Codigo_Vehiculo,Ccodigo_Victima,AnnoSiniestro,DD_Mes_Siniestro,hourSiniestro,minuSiniestro,Edad
count,2.802900e+04,2.802900e+04,25006.000000,28029.000000,28029.0,28029.000000,28029.000000,28029.000000,26519.000000
mean,1.407329e+07,1.115486e+07,1.458570,0.932962,2025.0,15.705733,12.674944,26.063577,36.314567
std,6.135940e+05,2.870518e+05,0.594266,1.531216,0.0,8.727175,5.933962,17.397723,16.069394
min,1.291689e+07,1.061423e+07,1.000000,0.000000,2025.0,1.000000,0.000000,0.000000,0.000000
25%,1.367136e+07,1.096760e+07,1.000000,0.000000,2025.0,8.000000,8.000000,10.000000,24.000000
50%,1.445433e+07,1.133342e+07,1.000000,1.000000,2025.0,16.000000,13.000000,30.000000,33.000000
75%,1.446175e+07,1.133650e+07,2.000000,1.000000,2025.0,23.000000,18.000000,40.000000,46.000000
max,1.447619e+07,1.134250e+07,9.000000,41.000000,2025.0,31.000000,23.000000,59.000000,104.000000


ACTOR_VIAL - describe() de texto


,Formulario,FechaSiniestro,mesSiniestro,Dia_Semana_Siniestro,Localidad,Sexo,Muerte_Posterior,Fecha_CambioGravedad,Gravedad_Indicador_30d,Condicion,Condicion_Especifica,Tipo_SITP,Con_Bicicleta,Con_Carga,Con_Moto,Con_Peaton,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Padron_Dual,victimaC,asociadoC
count,28029,28029,28029,28029,28027,26605,28029,212,28008,28029,28029,2898,3759,1878,18195,6140,618,337,3311,87,28029,28029
unique,11573,365,12,7,20,2,2,165,3,5,17,6,1,1,1,1,1,1,1,1,11,10
top,A001716215,2025-08-15,Septiembre,viernes,Kennedy,Masculino,N,2025-10-29,HERIDO,MOTOCICLISTA,Motociclista,ZONAL,SI,SI,SI,SI,SI,SI,SI,SI,Motociclistas,Motocicletas
freq,42,138,2563,4471,3659,19530,27810,4,19605,11131,8831,2148,3759,1878,18195,6140,618,337,3311,87,8831,9498


In [7]:
# Tabla de diagnóstico estructural
filas = []
for nombre, df in dfs.items():
    tipos = df.dtypes.astype(str).value_counts()
    resumen_tipos = ", ".join(f"{t}: {n}" for t, n in tipos.items())
    nulos = int(df.isna().sum().sum())
    filas.append({
        "Dataset": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Tipos de datos": resumen_tipos,
        "Observaciones": f"{nulos:,} celdas nulas; {int(df.duplicated().sum()):,} filas duplicadas",
    })
diagnostico_estructural = pd.DataFrame(filas)
diagnostico_estructural

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,mortalidad,13138,19,"str: 15, int64: 4",0 celdas nulas; 0 filas duplicadas
1,siniestros,11573,25,"str: 20, int64: 5","93,066 celdas nulas; 0 filas duplicadas"
2,vehiculos,19974,18,"str: 15, int64: 3","173,195 celdas nulas; 0 filas duplicadas"
3,actor_vial,28029,31,"str: 22, int64: 7, float64: 2","248,835 celdas nulas; 0 filas duplicadas"


**Características principales de cada dataset** _(escribe aquí tu interpretación a partir de las salidas anteriores: qué representa cada fila, qué columnas son clave para tu pregunta, qué tipos de datos predominan)._

## Momento 4. Diagnóstico de calidad
No se limpian los datos: solo se **diagnostica**, se interpreta y se documenta.

### 4.1 Valores faltantes

In [8]:
for nombre, df in dfs.items():
    faltantes = pd.DataFrame({
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
    })
    print(nombre.upper())
    display(faltantes[faltantes["nulos"] > 0] if faltantes["nulos"].any() else "Sin valores nulos")

MORTALIDAD


'Sin valores nulos'

SINIESTROS


,nulos,% nulos
Localidad,1,0.01
Elemento_Choque,3685,31.84
Tipo_Objeto_Fijo,11207,96.84
Con_Bicicleta,9856,85.16
Con_Carga,10867,93.90
Con_Moto,3997,34.54
Con_Peaton,8845,76.43
Con_Troncal,11355,98.12
Con_Alimentador,11437,98.82
Con_Zonal,10277,88.80


VEHICULOS


,nulos,% nulos
Servicio,2769,13.86
Modalidad,16801,84.11
Tipo_SITP,18252,91.38
Con_Bicicleta,16548,82.85
Con_Carga,18491,92.58
Con_Moto,5974,29.91
Con_Peaton,17087,85.55
Con_Troncal,19689,98.57
Con_Alimentador,19761,98.93
Con_Zonal,17903,89.63


ACTOR_VIAL


,nulos,% nulos
Codigo_Vehiculo,3023,10.79
Localidad,2,0.01
Edad,1510,5.39
Sexo,1424,5.08
Fecha_CambioGravedad,27817,99.24
Gravedad_Indicador_30d,21,0.07
Tipo_SITP,25131,89.66
Con_Bicicleta,24270,86.59
Con_Carga,26151,93.30
Con_Moto,9834,35.09


### 4.2 Tipos de datos y categorías

In [9]:
SENTINELAS = ["sin dato", "sin datos", "sin informacion", "sin información", "no reporta",
              "no aplica", "desconocido", "sin definir", "no registra", "n/a", "na"]

def parece_numerica(serie):
    s = serie.dropna().astype(str)
    return len(s) > 0 and pd.to_numeric(s, errors="coerce").notna().mean() > 0.95

for nombre, df in dfs.items():
    print("=" * 70)
    print(nombre.upper())
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        s = df[c].dropna().astype(str)
        if parece_numerica(df[c]):
            print(f"  [tipo] '{c}' guarda números como texto")
        k = s.str.strip().str.lower().isin(SENTINELAS).sum()
        if k:
            print(f"  [sentinela] '{c}' tiene {k:,} valores tipo 'sin dato / desconocido'")
        if s.nunique() != s.str.strip().str.upper().nunique():
            print(f"  [formato] '{c}' tiene categorías repetidas por mayúsculas/espacios")
        if any(p in c.lower() for p in ("fecha",)):
            print(f"  [tipo] '{c}' parece fecha pero está como texto (ej.: {s.iloc[0] if len(s) else ''})")
        if "hora" in c.lower() and not "rango" in c.lower():
            print(f"  [tipo] '{c}' parece hora y está como texto")

MORTALIDAD
  [sentinela] 'MES_DEL_HECHO' tiene 2 valores tipo 'sin dato / desconocido'
  [formato] 'MES_DEL_HECHO' tiene categorías repetidas por mayúsculas/espacios
  [sentinela] 'DIA_DEL_HECHO' tiene 2 valores tipo 'sin dato / desconocido'
  [formato] 'DIA_DEL_HECHO' tiene categorías repetidas por mayúsculas/espacios
  [sentinela] 'RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_' tiene 4,588 valores tipo 'sin dato / desconocido'
  [formato] 'RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_' tiene categorías repetidas por mayúsculas/espacios
  [sentinela] 'PERTENENCIA_GRUPAL' tiene 958 valores tipo 'sin dato / desconocido'
  [formato] 'PERTENENCIA_GRUPAL' tiene categorías repetidas por mayúsculas/espacios
  [sentinela] 'PAIS_DE_NACIMIENTO' tiene 64 valores tipo 'sin dato / desconocido'
  [formato] 'PAIS_DE_NACIMIENTO' tiene categorías repetidas por mayúsculas/espacios
  [sentinela] 'CLASE_O_TIPO_DE_ACCIDENTE_DE_TRANSPORTE' tiene 298 valores tipo 'sin dato / desconocido'
  [sentinela] 'CIRCUNSTANCIA_DEL_HECHO_

### 4.3 Valores de las variables clave (posibles inconsistencias)

In [10]:
# Categorías de las columnas con pocos valores distintos
for nombre, df in dfs.items():
    print("=" * 70)
    print(nombre.upper())
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 25:
            print(f"\n--- {c} ({df[c].nunique()} valores distintos) ---")
            print(df[c].value_counts(dropna=False).head(25))

MORTALIDAD

--- Sexo (2 valores distintos) ---
Sexo
Hombre    10310
Mujer      2828
Name: count, dtype: int64

--- MES_DEL_HECHO (25 valores distintos) ---
MES_DEL_HECHO
mayo               1014
marzo               976
octubre             974
junio               966
abril               932
febrero             884
julio               876
agosto              864
diciembre           860
septiembre          838
noviembre           808
enero               798
Diciembre           242
Septiembre          208
Octubre             206
Junio               206
Noviembre           202
Agosto              200
Febrero             196
Marzo               184
Mayo                182
Enero               182
Julio               170
Abril               168
Sin información       2
Name: count, dtype: int64

--- DIA_DEL_HECHO (15 valores distintos) ---
DIA_DEL_HECHO
sábado             1802
viernes            1670
jueves             1514
miércoles          1510
domingo            1478
martes             1430


In [11]:
# Vista previa NO limpia, solo para comprobar si los datos permiten responder la pregunta
# (se convierte 'casos' a número solo para sumar; no se modifica el DataFrame original)
if "casos" in dfs.get("mortalidad", pd.DataFrame()).columns:
    d = dfs["mortalidad"]
    casos = pd.to_numeric(d["casos"], errors="coerce")
    print("Valores de 'casos' no convertibles a número:", int(casos.isna().sum() - d["casos"].isna().sum()))
    for col in ["RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_", "DIA_DEL_HECHO", "CODIGO_LOCALIDAD",
                "MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE"]:
        if col in d.columns:
            print(f"\nMuertes (suma de casos) por {col}:")
            print(casos.groupby(d[col]).sum().sort_values(ascending=False).head(8))

Valores de 'casos' no convertibles a número: 0

Muertes (suma de casos) por RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_:
RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_
Sin información    4160
(18:00 a 20:59)     872
(21:00 a 23:59)     822
(06:00 a 08:59)     818
(15:00 a 17:59)     744
(12:00 a 14:59)     716
(09:00 a 11:59)     654
(03:00 a 05:59)     632
Name: casos, dtype: int64

Muertes (suma de casos) por DIA_DEL_HECHO:
DIA_DEL_HECHO
sábado       1802
viernes      1670
jueves       1514
miércoles    1510
domingo      1478
martes       1430
lunes        1386
Sábado        412
Name: casos, dtype: int64

Muertes (suma de casos) por CODIGO_LOCALIDAD:
CODIGO_LOCALIDAD
0     6569
8      868
10     596
11     480
19     452
9      419
7      414
16     402
Name: casos, dtype: int64

Muertes (suma de casos) por MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE:
MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE
No aplica          5086
Motocicleta        4988
Bicicleta          1650
Peatón              498
Automóvil           340
Sin

### 4.4 Tabla de hallazgos (borrador automático)
Revisa cada fila: **elimina** lo que no sea un problema real y **mejora** el impacto y la acción con tus palabras.

In [12]:
hallazgos = []
for nombre, df in dfs.items():
    n = len(df)
    for col, k in df.isna().sum().items():
        if k:
            hallazgos.append([nombre, f"Valores faltantes en '{col}'", f"{k:,} nulos ({k/n:.2%})",
                              "Sesgo en conteos y promedios; filas excluidas de agrupaciones",
                              "Decidir entre imputar, etiquetar como 'Sin dato' o excluir"])
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        s = df[c].dropna().astype(str)
        if parece_numerica(df[c]):
            hallazgos.append([nombre, f"'{c}' numérica almacenada como texto", f"dtype={df[c].dtype}",
                              "No se puede sumar ni promediar sin convertir",
                              "Convertir con pd.to_numeric(errors='coerce')"])
        k = s.str.strip().str.lower().isin(SENTINELAS).sum()
        if k:
            hallazgos.append([nombre, f"Categoría tipo 'sin dato' en '{c}'", f"{k:,} registros",
                              "Reducen la cobertura real de la variable",
                              "Tratarlos como faltantes y reportar el porcentaje"])
        if s.nunique() != s.str.strip().str.upper().nunique():
            hallazgos.append([nombre, f"Categorías duplicadas por formato en '{c}'", "nunique cambia al normalizar",
                              "Una misma categoría se contaría como varias",
                              "Normalizar con str.strip() y str.upper()"])
        if "fecha" in c.lower():
            hallazgos.append([nombre, f"'{c}' como texto", f"dtype={df[c].dtype}",
                              "No permite filtrar ni agrupar por fechas",
                              "Convertir con pd.to_datetime"])
    d = int(df.duplicated().sum())
    if d:
        hallazgos.append([nombre, "Filas duplicadas", f"{d:,} filas",
                          "Pueden inflar los conteos (o ser legítimas si los datos están agregados)",
                          "Revisar antes de eliminar"])

tabla_hallazgos = pd.DataFrame(hallazgos, columns=["Dataset", "Problema identificado", "Evidencia encontrada",
                                                   "Posible impacto", "Acción propuesta"])
tabla_hallazgos

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,mortalidad,Categoría tipo 'sin dato' en 'MES_DEL_HECHO',2 registros,Reducen la cobertura real de la variable,Tratarlos como faltantes y reportar el porcentaje
1,mortalidad,Categorías duplicadas por formato en 'MES_DEL_HECHO',nunique cambia al normalizar,Una misma categoría se contaría como varias,Normalizar con str.strip() y str.upper()
2,mortalidad,Categoría tipo 'sin dato' en 'DIA_DEL_HECHO',2 registros,Reducen la cobertura real de la variable,Tratarlos como faltantes y reportar el porcentaje
3,mortalidad,Categorías duplicadas por formato en 'DIA_DEL_HECHO',nunique cambia al normalizar,Una misma categoría se contaría como varias,Normalizar con str.strip() y str.upper()
4,mortalidad,Categoría tipo 'sin dato' en 'RANGO_DE_HORA_DEL_HECHO_XXX3_HORAS_',"4,588 registros",Reducen la cobertura real de la variable,Tratarlos como faltantes y reportar el porcentaje
...,...,...,...,...,...
59,actor_vial,Valores faltantes en 'Con_Alimentador',"27,692 nulos (98.80%)",Sesgo en conteos y promedios; filas excluidas de agrupaciones,"Decidir entre imputar, etiquetar como 'Sin dato' o excluir"
60,actor_vial,Valores faltantes en 'Con_Zonal',"24,718 nulos (88.19%)",Sesgo en conteos y promedios; filas excluidas de agrupaciones,"Decidir entre imputar, etiquetar como 'Sin dato' o excluir"
61,actor_vial,Valores faltantes en 'Con_Padron_Dual',"27,942 nulos (99.69%)",Sesgo en conteos y promedios; filas excluidas de agrupaciones,"Decidir entre imputar, etiquetar como 'Sin dato' o excluir"
62,actor_vial,'FechaSiniestro' como texto,dtype=str,No permite filtrar ni agrupar por fechas,Convertir con pd.to_datetime


### 4.5 Limitaciones frente a la pregunta analítica
## 4.5 Limitaciones frente a la pregunta analítica

1. **Cobertura distinta de las fuentes.** `mortalidad` solo registra víctimas fatales y cubre 2015–2026 (2026 incompleto, con 369 víctimas). El Anuario cubre solo 2025 y, según su variable de gravedad (solo "Con Heridos" y "Con Muertos"), parece no incluir siniestros con únicamente daños materiales. Los resultados valen para estos registros y no para todos los accidentes de Bogotá.
2. **La hora en `mortalidad` es poco útil.** Viene en rangos de 3 horas y el 34,9 % de los registros dice "Sin información". Para responder "a qué horas" conviene usar `Hora_Siniestro` del Anuario, que es exacta (0 a 23) y no tiene valores ausentes.
3. **Duplicación en `mortalidad`.** Cada víctima aparece dos veces (una con localidad "Bogotá" y otra con su localidad): 13.138 filas equivalen a 6.569 víctimas. Hay que eliminar la copia antes de contar muertes.
4. **La zona solo llega al nivel de localidad.** Ninguna fuente tiene barrio. El Anuario trae coordenadas (convertibles y todas dentro de Bogotá), pero `mortalidad` no. Además, los nombres de las localidades se escriben distinto en cada fuente ("CANDELARIA", "Candelaria", "La Candelaria") y en `mortalidad` el 4 % de las víctimas tiene "Sin dato".
5. **El tipo de vehículo se mide de forma distinta.** En `mortalidad` es el medio de desplazamiento de la víctima, y los peatones aparecen con dos etiquetas ("No aplica" y "Peatón"). En el Anuario hay que usar `Clase` en `vehiculos`, porque las banderas `Con_*` describen el siniestro y no el vehículo. Las categorías de vehículos también se solapan (motocicleta, motociclo, mototriciclo, motocarro).
6. **No hay un identificador común entre `mortalidad` y el Anuario.** No se pueden unir registro a registro, solo comparar totales por año, día, hora o localidad. Además las definiciones difieren (612 víctimas en `mortalidad` y 562 muertos en el Anuario para 2025), y los vocabularios también ("Hombre/Mujer" frente a "Masculino/Femenino").
7. **Los nulos de las columnas `Con_*`, `Modalidad` y `Fecha_CambioGravedad` significan "no aplica".** Hay que interpretarlos así para no subestimar la cobertura real.
8. **Calidad del texto.** Mayúsculas y formatos mezclados (por ejemplo, 2.346 filas de `mortalidad` con mes y día capitalizados) parten los conteos si no se normalizan.

Conclusión: la pregunta se puede responder con el Anuario 2025 para hora, día, localidad y tipo de vehículo, y con `mortalidad` (sin duplicados y filtrada por año) para ver la evolución de las muertes y cruzar con 2025. Antes hay que aplicar la limpieza propuesta.

## Momento 5. Informe y documentación
El informe técnico está en `reports/informe_diagnostico.md` y el `README.md` explica cómo reproducir este diagnóstico.

## Momento 6. Verificación
- [ ] Reiniciar el kernel y ejecutar todo sin errores
- [ ] Todos los datasets cargados y diagnosticados
- [ ] El informe coincide con estos resultados
- [ ] `requirements.txt` y `README.md` actualizados
- [ ] Commits y push a GitHub; repositorio accesible para el instructor

In [14]:
mort, sin, veh, act = dfs["mortalidad"], dfs["siniestros"], dfs["vehiculos"], dfs["actor_vial"]
print("Valores de casos:", mort["casos"].unique())
print(mort["ANO"].value_counts().sort_index())
loc_ok = ~mort["LOCALIDAD"].str.strip().str.lower().isin(["bogotá", "sin dato"])
print(f"Registros con localidad específica: {loc_ok.mean():.1%}")
print(pd.crosstab(mort["MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE"], mort["CONDICION_DE_LA_VICTIMA_AT_"]))
cs, cv, ca = set(sin["Codigo_Siniestro"]), set(veh["Codigo_Siniestro"]), set(act["Codigo_Siniestro"])
print("Siniestros únicos:", sin["Codigo_Siniestro"].nunique(), "de", len(sin))
print("En vehículos pero no en siniestros:", len(cv - cs), "| en actor_vial pero no en siniestros:", len(ca - cs))
print("Siniestros sin vehículo:", len(cs - cv), "| sin actor vial:", len(cs - ca))
print(pd.crosstab(act["Muerte_Posterior"], act["Fecha_CambioGravedad"].notna()))
lat = pd.to_numeric(sin["Latitud"].str.replace(",", "."), errors="coerce")
lon = pd.to_numeric(sin["Longitud"].str.replace(",", "."), errors="coerce")
print("Coordenadas no convertibles:", int(lat.isna().sum()), int(lon.isna().sum()))
fuera = ~(lat.between(3.7, 4.95) & lon.between(-74.5, -73.9))
print("Coordenadas fuera del rango aproximado de Bogotá:", int(fuera.sum()))
print(sin["Hora_Siniestro"].value_counts().sort_index().to_string())

Valores de casos: [1]
ANO
2015    1112
2016    1182
2017    1106
2018    1054
2019    1046
2020     812
2021    1016
2022    1260
2023    1258
2024    1330
2025    1224
2026     738
Name: count, dtype: int64
Registros con localidad específica: 48.0%
CONDICION_DE_LA_VICTIMA_AT_           Conductor  Pasajero  Peatón  \
MEDIO_DE_DESPLAZAMIENTO_O_TRANSPORTE                                
Automóvil                                   164       176       0   
Avión – avioneta                              2         6       0   
Avión, Avioneta                               2         6       0   
Bicicleta                                  1618        32       0   
Bicimotor                                     2         0       0   
Bicitaxi                                     10         4       0   
Bus                                          36        78       0   
Buseta                                        2        26       0   
Camioneta                                    66        32   